# PickLabel on a GPU (Colab / Kaggle T4)

The GPU half of the loop: SAM 3 baseline, SAM 3 + SAM 2.1 auto-labelling, student training, evaluation and latency.
The CPU half (capture, alignment, change maps, Roboflow sync) runs on a laptop.

Needs: a GPU runtime, approved access to [facebook/sam3](https://huggingface.co/facebook/sam3) on Hugging Face, and (optionally) a Roboflow API key.

In [ ]:
!git clone https://github.com/Waqar53/picklable.git && cd picklable && pip install -q -e ".[models,roboflow]"
%cd picklable

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (with facebook/sam3 access): ")
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key (Enter to skip): ")
os.environ["ROBOFLOW_WORKSPACE"] = "your-workspace"   # edit
os.environ["ROBOFLOW_PROJECT"] = "grip-contamination"

In [ ]:
from huggingface_hub import hf_hub_download
sam3 = hf_hub_download("facebook/sam3", "sam3.pt", local_dir="weights", token=os.environ["HF_TOKEN"])
print(sam3)

## 1. SAM 3 zero-shot baseline: prompt sets x threshold (on the WS2 conveyor test set, then on your gold set)

In [ ]:
!picklabel download ws2 --out data/raw
!picklabel sam3-baseline --weights weights/sam3.pt --gt data/raw/ws2/ws2_test_coco.json --out report/sam3_ws2
# gold set exported from Roboflow as COCO:
# !picklabel sam3-baseline --weights weights/sam3.pt --gt data/gold/_annotations.coco.json --out report/sam3_gold

## 2. Auto-label your pick pairs with SAM 3 candidates + SAM 2.1 grasp prompts, push to Roboflow

In [ ]:
# upload your pairs (zip of data/pairs from the rig) first, then:
!picklabel autolabel data/pairs --out runs/autolabel/home --sam3 weights/sam3.pt --sam2 sam2.1_b.pt --origin home --push
!picklabel eval-autolabel runs/autolabel/home --out report/autolabel   # needs gold masks on some picks

## 3. Train students (R2: accepted labels only, R3: accepted + reviewed)

In [ ]:
!picklabel build-dataset runs/autolabel/home --out datasets/R2 --routes accept
!picklabel build-dataset runs/autolabel/home --out datasets/R3 --routes accept,review
!picklabel train --data datasets/R2/data.yaml --preset R2 --labeling-minutes 0
!picklabel train --data datasets/R3/data.yaml --preset R3 --labeling-minutes 35   # stopwatch minutes spent reviewing

## 4. Evaluate on the gold set (same metrics as SAM 3) and time on the T4

In [ ]:
for run in ["R2", "R3"]:
    !picklabel eval-student --weights runs/train/{run}/weights/best.pt --gt data/gold/_annotations.coco.json --out report/eval/{run} --name {run} --latency --device 0

In [ ]:
!picklabel report runs.yaml --out report   # see picklabel/eval/report.py for the runs.yaml format